In [44]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import collections
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset

from src.dataset import make_loaders, simulate_imbalance
from src.evaluate import predict_loader, summarize
from src.experiment import run_experiment, save_result
from src.model import CNNBaseline
from src.sampler import BalancedBatchSampler
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()



train_ds, _, train_loader, val_loader = make_loaders(DATA, IMG_SIZE, BATCH, SEED)
print(train_ds.class_to_idx)

keep_fraction = {0: 0.2, 4: 0.3}   
keep = simulate_imbalance(train_ds.targets, keep_fraction, SEED)
imb_ds = Subset(train_ds, keep)
imb_targets = [train_ds.targets[i] for i in keep]
print(pd.Series(imb_targets).value_counts().sort_index())

pd.DataFrame({'index': keep, 'path': [train_ds.samples[i][0] for i in keep]}) \
  .to_csv('../data/processed/imbalanced_train_indices.csv', index=False)

paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
{'ambulance': 0, 'autobus': 1, 'kamyun': 2, 'kamyunet': 3, 'minibus': 4, 'savari': 5, 'taxi': 6, 'vanet': 7}
0     35
1    196
2    200
3    191
4     54
5    198
6    199
7    199
Name: count, dtype: int64


In [39]:
sampler = BalancedBatchSampler(imb_targets , 32 , num_batches=10 , seed=SEED)
batch=next(iter(sampler))
print(collections.Counter(imb_targets[i] for i in batch))

Counter({1: 4, 6: 4, 5: 4, 2: 4, 0: 4, 4: 4, 7: 4, 3: 4})


In [45]:
first_row = run_experiment('imb_standard', device , paths ,  train_subset=keep, sampler_type='none')
second_row = run_experiment('imb_balanced', device , paths, train_subset=keep, sampler_type='balanced')
print(first_row)
print(second_row)

{'epoch': 1, 'train_loss': 1.8860882565660297, 'val_loss': 2.104246766226632, 'val_macro_f1': 0.20347210396715348, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.7069787469300084, 'val_loss': 1.9966856083312592, 'val_macro_f1': 0.2162751887770957, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.644104373530022, 'val_loss': 1.9542132238288978, 'val_macro_f1': 0.281596873715098, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.5885950371904194, 'val_loss': 1.89566564250302, 'val_macro_f1': 0.3009882499374242, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.5209407034160207, 'val_loss': 1.7399937474882448, 'val_macro_f1': 0.32981523095546067, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.4620717464003172, 'val_loss': 1.706644619749738, 'val_macro_f1': 0.32213016365268904, 'lr': 0.001}
{'epoch': 7, 'train_loss': 1.4138228728336357, 'val_loss': 1.7153744307431307, 'val_macro_f1': 0.33586906704666497, 'lr': 0.001}
{'epoch': 8, 'train_loss': 1.3791849500728104, 'val_loss': 1.6487649182994644, 'val_macro_f1': 0.376488397